# 1. Setup

Member D: CLIP ViT-B/16. Five experiments: zero-shot, Partial-1/3/6 and Full. One prompt is used throughout. Only validation chooses checkpoints and partial depth; smaller depth wins a tie. Final tests run after all choices are fixed. `DEBUG=True` uses shared small subsets and one epoch. DEBUG filenames are separate from formal results. No training results have been produced during authoring.

API inspection: implemented common interfaces inspected at team master `05d499e`. Pair rows are keyed by crop/disease, not raw CSV composition IDs. The shared pair table reads test metadata for candidate names; test images and metrics must not be used in training or selection. Shared validation is a per-pair 10% sample of train.csv, not held-out compositions. Requirements pin OpenCLIP 3.3.0. All common files remain owned by Member E.

This expanded experiment notebook exceeds the 100-line soft limit to support five runs, checks and provenance; it remains below the 150-line hard limit. Discuss this exception with Member E during review. Use Python 3.11 per team brief.

In [ ]:
import gc, json, sys, time
from pathlib import Path
from importlib.metadata import version
import open_clip, torch
import torch.nn.functional as F
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'common/dataset.py').is_file())
sys.path.insert(0, str(ROOT))
from common import config as cfg
from common.dataset import get_loaders, get_pairs, get_crop_and_disease_names, load_split
from common.evaluate import evaluate, val_accuracy
MODEL_NAME, DEBUG = 'approach3_clip', True
HP = dict(backbone='ViT-B-16', pretrained='openai', seed=42, lr=1e-5, weight_decay=0.05,
          epochs=10, batch_size=8, optimizer='AdamW', partial_depths=[1, 3, 6],
          prompt='a photo of a {crop} leaf with {disease}', norm='clip', robust=True,
          gradient_checkpointing=True, selection='validation_top1', tie_break='smaller_depth')
cfg.set_seed(HP['seed'])
RUN = MODEL_NAME + ('_debug' if DEBUG else '')
OUT = cfg.CHECKPOINT_DIR / RUN
OUT.mkdir(parents=True, exist_ok=True)
environment = {p: version(p) for p in ['torch', 'torchvision', 'open_clip_torch']}
environment.update(python=sys.version, device=str(cfg.DEVICE), debug=DEBUG)


# 2. Data

Use actual shared `load_split`, `get_pairs`, `get_crop_and_disease_names` and `get_loaders`; no substitute dataset or mapping. Check full CSV counts before DEBUG subsetting. Validation and train images are disjoint. Unseen text names are permitted side information. File checks require the complete local DPD-Small image collection, even for DEBUG.

In [ ]:
frames = [load_split(p) for p in [cfg.TRAIN_CSV, cfg.TEST_SEEN_CSV, cfg.TEST_UNSEEN_CSV]]
assert [len(f) for f in frames] == [8000, 2000, 1448], 'Check the actual DPD-Small CSVs.'
pairs = get_pairs()
assert pairs.shape == (cfg.NUM_PAIRS, 2) and cfg.NUM_PAIRS == 333 and cfg.NUM_SEEN_PAIRS == 323
assert len(set(map(tuple, pairs.tolist()))) == cfg.NUM_PAIRS
assert set(frames[0].pair_row) == set(range(cfg.NUM_SEEN_PAIRS))
assert set(frames[1].pair_row) == set(range(cfg.NUM_SEEN_PAIRS))
assert set(frames[2].pair_row) == set(range(cfg.NUM_SEEN_PAIRS, cfg.NUM_PAIRS))
assert all(not set(a.image) & set(b.image) for i, a in enumerate(frames) for b in frames[i+1:])
crop_names, disease_names = get_crop_and_disease_names()
prompts = [HP['prompt'].format(crop=crop_names[c].replace('_', ' '),
                              disease=disease_names[d].replace('_', ' ')) for c, d in pairs.tolist()]
def loaders():
    cfg.set_seed(HP['seed'])
    return get_loaders(HP['batch_size'], norm=HP['norm'], debug=DEBUG)
train_loader, val_loader, seen_loader, unseen_loader = loaders()
print('DataLoader workers:', train_loader.num_workers, val_loader.num_workers)
assert not set(train_loader.dataset.frame.image) & set(val_loader.dataset.frame.image)
sample_counts = dict(zip(['train', 'val', 'seen', 'unseen'],
                         [len(x.dataset) for x in [train_loader, val_loader, seen_loader, unseen_loader]]))
assert all((cfg.IMAGE_DIR / name).is_file() for f in frames for name in f.image), 'Missing image files.'
for key in ['crop', 'disease']:
    assert all(f.groupby(key)[key+'_name'].nunique().max() == 1 for f in frames), 'Inconsistent concept names.'

# 3. Model

OpenCLIP matching tokenizer and reference transform are loaded. Shared CLIP normalisation is checked; shared resizing/augmentation remains the team protocol, rather than replacing it with the reference transform. `force_quick_gelu=True` matches the OpenAI checkpoint activation. API reference: https://github.com/mlfoundations/open_clip/tree/v3.3.0

Image/text embeddings are L2-normalised; scaled cosine scores have shape `[B,333]`. The partial helper inspects the real named module and verifies the exact trainable parameter identities: only last N visual blocks. Output projection and text remain frozen. Model inspection fails clearly if the architecture differs. Frozen towers stay in eval mode while trainable blocks still receive gradients. This model has no train-mode-only layer required for partial optimisation.

Smoke test checks one real validation image batch, text token count, score shape and finiteness. It checks freeze configurations before training. Initial weights are saved locally so every run resets to identical pretrained parameters.

In [ ]:
model, _, reference_transform = open_clip.create_model_and_transforms(
    HP['backbone'], pretrained=HP['pretrained'], force_quick_gelu=True)
model = model.to(cfg.DEVICE)
model.set_grad_checkpointing(HP['gradient_checkpointing'])
tokenizer = open_clip.get_tokenizer(HP['backbone'])
tokens = tokenizer(prompts).to(cfg.DEVICE)
assert tokens.shape[0] == cfg.NUM_PAIRS
assert tuple(reference_transform.transforms[-1].mean) == cfg.NORM_STATS['clip'][0]
torch.save(model.cpu().state_dict(), OUT / 'initial.pt')
model.to(cfg.DEVICE)
def freeze_last_n_visual_blocks(n):
    assert n in HP['partial_depths']
    blocks = dict(model.visual.named_modules()).get('transformer.resblocks')
    if not isinstance(blocks, torch.nn.ModuleList) or len(blocks) < n:
        raise RuntimeError('Unexpected OpenCLIP visual blocks; inspect model.visual.')
    model.requires_grad_(False)
    for block in list(blocks)[-n:]:
        block.requires_grad_(True)
    expected = {id(p) for block in list(blocks)[-n:] for p in block.parameters()}
    assert {id(p) for p in model.parameters() if p.requires_grad} == expected
    return [name for name, p in model.named_parameters() if p.requires_grad]
def text_embeddings():
    return F.normalize(model.encode_text(tokens).float(), dim=1)
@torch.no_grad()
def predict(images):
    model.eval()
    text = text_embeddings()
    return model.logit_scale.exp() * F.normalize(model.encode_image(images.to(cfg.DEVICE)).float(), dim=1) @ text.T
model.eval()
images, _, _, labels = next(iter(val_loader))
scores = predict(images)
assert scores.shape == (len(images), cfg.NUM_PAIRS) and torch.isfinite(scores).all()
assert scores.argmax(1).shape == labels.shape
for depth in HP['partial_depths']:
    print('Partial', depth, 'trainable tensors:', len(freeze_last_n_visual_blocks(depth)))
print('Smoke shapes:', images.shape, tokens.shape, scores.shape)

# 4. Training

Five independent runs reset pretrained weights, seed and loaders. CE uses seen text features only. Partial text features are cached with no gradients. Full re-encodes prompts every training batch and asserts a text embedding gradient. Gradient checkpointing reduces school GPU memory use. Training uses float32 for portability; batch size 8 is a conservative default and must be fixed before full experiments.

Best checkpoints use shared validation Top-1 over all 333 candidates. Training loss, validation, runtime, parameter names/counts and prompt mapping are retained. Training histories for all epochs are in selection.json. Checkpoints retain history up to the selected epoch. No early stopping or seen-score bias is used. Zero-shot has zero trainable parameters. The initial DEBUG run exercises actual training/validation and checkpoint saves before formal training; its outputs are not report results.

### Optional local training smoke test

Run this cell to test Partial-1, Partial-3, Partial-6 and Full on **one image / one update each**. It checks finite loss/gradients and an actual weight update, then restores initial weights. Full also checks text gradients. It does not run epochs or final tests. CPU Full may take noticeably longer. Run it after Model setup and before formal training. The following large Training cell still runs the original five-experiment workflow; stop here when checking your laptop.


In [ ]:
from approaches.clip_smoke import run_training_smoke
smoke_reports = run_training_smoke(model, tokens, train_loader, cfg, HP, freeze_last_n_visual_blocks, OUT / 'initial.pt')

In [ ]:
def fit(name, names, frozen_text):
    history, best = [], -1.0
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=HP['lr'], weight_decay=HP['weight_decay'])
    for epoch in range(1 if DEBUG else HP['epochs']):
        started = time.perf_counter()
        loss, score = train_epoch(optimizer, frozen_text), val_accuracy(predict, val_loader)
        history.append(dict(epoch=epoch+1, loss=loss, val_top1=score, seconds=time.perf_counter()-started))
        print(name, history[-1])
        if score > best:
            best = score
            save_checkpoint(name, history, best, names)
    return best, history
def save_checkpoint(name, history, best, names):
    torch.save(dict(model_name=name, state_dict={k: v.detach().cpu() for k, v in model.state_dict().items()},
                    hp=HP, environment=environment, prompts=prompts, pairs=pairs,
                    history=history, best_val=best, trainable_names=names,
                    trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad)),
               OUT / (name + '.pt'))
def train_epoch(optimizer, frozen_text):
    model.train()
    if frozen_text is not None:
        model.eval()  # frozen towers retain their pretrained inference behaviour
    total, count = 0.0, 0
    for batch, _, _, target in train_loader:
        optimizer.zero_grad(set_to_none=True)
        text = text_embeddings()[:cfg.NUM_SEEN_PAIRS] if frozen_text is None else frozen_text[:cfg.NUM_SEEN_PAIRS]
        logits = model.logit_scale.exp() * F.normalize(model.encode_image(batch.to(cfg.DEVICE)).float(), dim=1) @ text.T
        loss = F.cross_entropy(logits, target.to(cfg.DEVICE))
        assert torch.isfinite(loss)
        loss.backward()
        assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters() if p.requires_grad)
        if frozen_text is None:
            assert model.token_embedding.weight.grad is not None, 'Text encoder must receive gradients.'
        optimizer.step()
        total, count = total + loss.item() * len(batch), count + len(batch)
    return total / count
def run_experiment(kind):
    global train_loader, val_loader, seen_loader, unseen_loader
    train_loader, val_loader, seen_loader, unseen_loader = loaders()
    model.load_state_dict(torch.load(OUT / 'initial.pt', map_location='cpu', weights_only=True))
    model.requires_grad_(kind == 'full')
    names = freeze_last_n_visual_blocks(kind) if isinstance(kind, int) else [k for k,p in model.named_parameters() if p.requires_grad]
    name, history, best = RUN + '_' + str(kind), [], -1.0
    model.eval()
    with torch.no_grad():
        frozen_text = None if kind == 'full' else text_embeddings()
    if kind == 'zero':
        save_checkpoint(name, history, val_accuracy(predict, val_loader), names)
        return name, None
    best, history = fit(name, names, frozen_text)
    return name, dict(depth=kind, val_top1=best, history=history)
zero_name, _ = run_experiment('zero')
partial = [run_experiment(depth) for depth in HP['partial_depths']]
best_partial = max(partial, key=lambda item: (item[1]['val_top1'], -item[1]['depth']))[0]
full_name, full_history = run_experiment('full')
selection = dict(best_partial=best_partial, candidates=dict(partial), full=full_history,
                 metric=HP['selection'], tie_break=HP['tie_break'], debug=DEBUG)
(OUT / 'selection.json').write_text(json.dumps(selection, indent=2))

# 5. Evaluation

Run only after the partial selection file is saved. Final comparison: Zero-shot vs selected Partial vs Full. Reload each best checkpoint, re-encode text using its final state, then use common evaluation including robustness. Every experiment has a distinct filename. Shared confusion PNG and CSV provide class-level plots. Robustness simulates brightness, blur and noise on seen images; it is not a separate field-image benchmark. Validation does not estimate unseen generalisation; discuss this limitation in the report.

In [ ]:
reports = []
for name in [zero_name, best_partial, full_name]:
    checkpoint = torch.load(OUT / (name + '.pt'), map_location='cpu', weights_only=True)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    details = dict(HP, environment=environment, samples=sample_counts,
                   trainable_parameters=checkpoint['trainable_parameters'], best_val=checkpoint['best_val'])
    reports.append(evaluate(predict, seen_loader, unseen_loader, name, details, robust=HP['robust']))


# 6. Save

Checkpoint files remain under checkpoints/approach3_clip[_debug]/. Final metric JSON, comparison JSON and shared confusion figures are under results/. Files include run settings, counts, prompt order and environment; use the recorded order for app integration. Initial weights and downloaded model caches are local only. No app adapter is modified here.

Restart kernel and Run All after setting DEBUG=False, only after DEBUG succeeds. See member_d_school_run.md for school setup and handoff. Commit notebook with outputs cleared; preserve experiment artifacts separately. Authoring checks cover notebook JSON, Python compilation and isolated freeze-helper behavior; real OpenCLIP/GPU/data execution remains pending on the school computer.

In [ ]:
summary = dict(selection=selection, results=reports, hp=HP, environment=environment, samples=sample_counts)
(cfg.RESULTS_DIR / (RUN + '_comparison.json')).write_text(json.dumps(summary, indent=2))
